In [1]:
import pandas as pd
from copy import copy

pd.options.display.max_columns = 37
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)

In [11]:
stats = pd.read_csv('csv/stats/RU1_players_24-25.csv')
values = pd.read_csv('csv/values/PO1_players_values_24-25.csv')

In [2]:
changes = {
    'é':'e',
    'ç':'c',
    'Ї':'I',
    'ć':'c',
    'ě':'e',
    'á':'a',
    'ä':'a',
    'ñ':'n',
    'ô':'o',
    'Ž':'Z',
    'ë':'e',
    'ã':'a',
    'ł':'l',
    'ś':'s',
    'Ó':'O',
    'š':'s',
    'Đ':'Dj',
    'ü':'u',
    'Á':'A',
    'ú':'u',
    'â':'a',
    'É':'E',
    'ê':'e',
    'í':'i',
    'ö':'o',
    'č':'c',
    'Ñ':'N',
    'Ö':'O',
    'ø':'o',
    'Ü':'U',
    'ę':'e',
    'ğ':'g',
    'ı':'i',
    'Þ':'Th',
    'è':'e',
    'ð':'d',
    'ý':'y',
    'Š':'S',
    'Ć':'C',
    'ž':'z',
    'ş':'s',
    'đ':'dj',
    'ò':'o',
    'ș':'s',
    'ă':'a',
    'Ç':'C',
    'Č':'C',
    'æ':'ae',
    'ń':'n',
    'ă':'a',
    'ț':'t',
    'Ł':'L',
    'ì':'i',
    'Ż':'Z',
    'ź':'z',
    'ō':'o',
    'ß':'ss',
    'ř':'r',
    'å':'a',
    'À':'A',
    'Ú':'U',
    'ů':'u',
    'õ':'o',
    'ū':'u',
    'ó':'o',
    'ï':'i',
    '\xad':''
}

champ_ids = ['BE1', 'BRA1', 'ES1', 'FR1', 'GB1', 'IT1', 'L1', 'NL1', 'PO1', 'RU1']

In [2]:
def unite_duplicates(stats):
    pairs_to_unite = []
    for i in stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep='last')].index:
        pairs_to_unite.append((i, i+1))
    duplicated_players = stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep=False)]
    stats = stats.drop_duplicates(subset=['Player', 'Age', 'Nation'], keep='last')
    for pair in pairs_to_unite:
        for col in stats.columns[5:24]:
            stats.at[pair[1], col] = duplicated_players.at[pair[0], col] + duplicated_players.at[pair[1], col]
        stats.at[pair[1], 'Gls.1'] = stats.at[pair[1], 'Gls']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'Ast.1'] = stats.at[pair[1], 'Ast']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'G+A.1'] = stats.at[pair[1], 'G+A']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'G-PK.1'] = stats.at[pair[1], 'G-PK']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'G+A-PK'] = (stats.at[pair[1], 'G-PK'] + stats.at[pair[1], 'Ast'])/stats.at[pair[1], '90s']
        stats.at[pair[1], 'xG.1'] = stats.at[pair[1], 'xG']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'xAG.1'] = stats.at[pair[1], 'xAG']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'xG+xAG'] = (stats.at[pair[1], 'xAG'] + stats.at[pair[1], 'xG'])/stats.at[pair[1], '90s']
        stats.at[pair[1], 'npxG.1'] = stats.at[pair[1], 'npxG']/stats.at[pair[1], '90s']
        stats.at[pair[1], 'npxG+xAG.1'] = stats.at[pair[1], 'npxG+xAG']/stats.at[pair[1], '90s']
        if pd.isna(stats.at[pair[1], 'Value']):
            stats.at[pair[1], 'Value'] = duplicated_players.at[pair[0], 'Value']
    cols = list(stats.columns)
    for i in range(cols.index('Gls.1'), len(cols)-1):
        stats[cols[i]] = stats[cols[i]].round(2)
    return stats
            

def do_join(stats, values):
    vals = copy(values)
    for letter in changes:
        stats['Player'] = stats['Player'].str.replace(letter, changes[letter])
        vals['Player'] = vals['Player'].str.replace(letter, changes[letter])
    stats['Value'] = [pd.NA]*len(stats)
    for i in range(len(stats)):
        for j in range(len(vals)):
            if stats.at[i, 'Player']==vals.at[j,'Player'] and stats.at[i,'Squad']==vals.at[j, 'Squad']:
                stats.at[i, 'Value'] = vals.at[j, 'Value']
                vals.drop(j)
                j-=1
    na_indexes = stats[stats['Value'].isna()].index
    for i in na_indexes:
        names = stats.at[i,'Player'].split(' ')
        for j in range(len(vals)):
            if stats.at[i,'Squad']==vals.at[j, 'Squad'] and any(name == vals.at[j,'Player'] for name in names):
                stats.at[i, 'Value'] = vals.at[j, 'Value']
                vals.drop(j)
                j-=1
    na_indexes = stats[stats['Value'].isna()].index
    for i in na_indexes:
        for j in range(len(vals)):
            names = vals.at[j,'Player'].split(' ')
            if stats.at[i, 'Squad']==vals.at[j,'Squad'] and any(stats.at[i,'Player'] == name for name in names):
                stats.at[i,'Value'] = vals.at[j,'Value']
                vals.drop(j)
                j-=1

In [92]:
merged_stats = stats.merge(values, on=['Player', 'Squad'], how='left')
print("количество null-стоимостей - ", len(merged_stats[merged_stats['Value'].isna()]))
merged_stats

количество null-стоимостей -  173


,Player,Nation,Pos,Squad,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,PrgR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,xG.1,xAG.1,xG+xAG,npxG.1,npxG+xAG.1,Value
0,Rodrigo Abascal,URU,DF,Boavista,30,29,29,2599,28.9,0,0,0,0,0,0,8,1,0.8,0.8,1.7,2.5,10,66,24,0.00,0.00,0.00,0.00,0.00,0.03,0.06,0.09,0.03,0.09,0.700
1,Nelson Abbey,ENG,DF,Rio Ave,20,8,7,635,7.1,0,0,0,0,0,0,1,0,0.1,0.1,0.1,0.2,7,14,7,0.00,0.00,0.00,0.00,0.00,0.01,0.02,0.03,0.01,0.03,0.700
2,Bilal Abdelrahman Zeeni,EGY,MF,Estrela,21,1,0,8,0.1,0,0,0,0,0,0,0,0,0.0,0.0,0.1,0.1,0,1,1,0.00,0.00,0.00,0.00,0.00,0.18,0.81,0.98,0.18,0.98,NaN
3,Giorgi Aburjania,GEO,MF,AVS Futebol,29,9,3,306,3.4,0,0,0,0,0,0,0,0,0.3,0.3,0.6,0.9,3,22,4,0.00,0.00,0.00,0.00,0.00,0.09,0.17,0.26,0.09,0.26,0.400
4,Adriano,BRA,MF,Santa Clara,24,32,31,2630,29.2,0,2,2,0,0,0,8,1,1.0,1.0,2.7,3.7,26,99,28,0.00,0.07,0.07,0.00,0.07,0.04,0.09,0.13,0.04,0.13,NaN
5,Lucas Áfrico,BRA,DF,Farense,29,21,20,1757,19.5,1,0,1,1,0,0,6,0,1.3,1.3,0.1,1.3,6,31,3,0.05,0.00,0.05,0.05,0.05,0.07,0.00,0.07,0.07,0.07,0.300
6,Salvador Agra,POR,"MF,DF",Boavista,32,32,31,2731,30.3,1,2,3,1,0,0,9,1,1.4,1.4,4.1,5.5,56,73,147,0.03,0.07,0.10,0.03,0.10,0.05,0.14,0.18,0.05,0.18,0.200
7,Brandon Aguilera,CRC,"MF,FW",Rio Ave,21,25,15,1378,15.3,3,1,4,3,0,0,4,0,1.4,1.4,2.2,3.6,26,72,44,0.20,0.07,0.26,0.20,0.26,0.09,0.14,0.23,0.09,0.23,1.500
8,Jorge Aguirre,CUB,FW,Gil Vicente FC,24,31,16,1354,15.0,3,2,5,3,0,0,2,1,3.6,3.6,2.1,5.7,5,23,67,0.20,0.13,0.33,0.20,0.33,0.24,0.14,0.38,0.24,0.38,0.600
9,Babatunde Akinsola,NGA,"MF,FW",AVS Futebol,21,30,11,1170,13.0,2,1,3,2,0,0,1,0,2.9,2.9,0.8,3.6,43,33,88,0.15,0.08,0.23,0.15,0.23,0.22,0.06,0.28,0.22,0.28,NaN


In [93]:
do_join(stats, values)
stats = unite_duplicates(stats)
print("количество null-объектов =", len(stats[stats['Value'].isna()]))
stats[stats['Value'].isna()]

количество null-объектов = 83


C:\Users\denis\AppData\Local\Temp\ipykernel_4944\3803764476.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stats[cols[i]] = stats[cols[i]].round(2)


,Player,Nation,Pos,Squad,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,PrgR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,xG.1,xAG.1,xG+xAG,npxG.1,npxG+xAG.1,Value
2,Bilal Abdelrahman Zeeni,EGY,MF,Estrela,21,1,0,8,0.1,0,0,0,0,0,0,0,0,0.0,0.0,0.1,0.1,0,1,1,0.00,0.00,0.00,0.00,0.00,0.18,0.81,0.98,0.18,0.98,<NA>
9,Babatunde Akinsola,NGA,"MF,FW",AVS Futebol,21,30,11,1170,13.0,2,1,3,2,0,0,1,0,2.9,2.9,0.8,3.6,43,33,88,0.15,0.08,0.23,0.15,0.23,0.22,0.06,0.28,0.22,0.28,<NA>
13,Goncalo Almeida,POR,"DF,MF",Boavista,21,19,9,814,9.0,0,0,0,0,0,0,1,0,0.7,0.7,0.9,1.6,12,18,43,0.00,0.00,0.00,0.00,0.00,0.08,0.09,0.17,0.08,0.17,<NA>
17,Artur Jorge Marques Amorim,POR,DF,Farense,29,19,15,1281,14.2,0,1,1,0,0,0,9,0,0.9,0.9,0.3,1.2,1,22,1,0.00,0.07,0.07,0.00,0.07,0.07,0.02,0.09,0.07,0.09,<NA>
24,Maximiliano Araujo,URU,"DF,MF",Sporting CP,24,28,19,1771,19.7,3,4,7,3,0,0,7,1,2.6,2.6,4.3,6.9,52,89,201,0.15,0.20,0.36,0.15,0.36,0.13,0.22,0.35,0.13,0.35,<NA>
40,Ney Bahia,BRA,MF,Santa Clara,20,2,0,62,0.7,0,0,0,0,0,0,0,0,0.1,0.1,0.0,0.1,0,0,2,0.00,0.00,0.00,0.00,0.00,0.18,0.00,0.18,0.18,0.18,<NA>
49,Leandro Barreiro Martins,LUX,MF,Benfica,24,27,11,1211,13.5,3,1,4,3,0,0,5,0,3.8,3.8,1.6,5.4,12,50,50,0.22,0.07,0.30,0.22,0.30,0.28,0.12,0.40,0.28,0.40,<NA>
50,Gil Bastiao Dias,POR,"FW,MF",Famalicão,27,30,13,1406,15.6,5,2,7,5,0,0,7,0,2.8,2.8,1.1,3.8,45,61,123,0.32,0.13,0.45,0.32,0.45,0.18,0.07,0.24,0.18,0.24,<NA>
56,Bachir Belloumi,ALG,FW,Farense,22,3,3,270,3.0,0,0,0,0,0,0,0,0,0.2,0.2,0.1,0.3,8,14,20,0.00,0.00,0.00,0.00,0.00,0.08,0.02,0.10,0.08,0.10,<NA>
58,Fahem Benaissa Yahia,FRA,"MF,DF",Casa Pia,22,8,4,386,4.3,1,0,1,1,0,0,1,0,0.3,0.3,0.0,0.3,10,10,19,0.23,0.00,0.23,0.23,0.23,0.07,0.00,0.07,0.07,0.07,<NA>


In [387]:
values[values['Player'].str.contains('Edney')]

,Player,Value,Squad
356,Edney,0.3,Santa Clara


In [388]:
stats.at[40, 'Value'] = 0.3

In [385]:
stats.at[570, 'Player']

'Marious Vrousai'

In [389]:
stats.to_csv('csv/stats/PO1_players_24-25.csv', index=False)

In [21]:
leagues = ['BE1', 'BRA1', 'GB1', 'ES1', 'FR1', 'IT1', 'L1', 'NL1', 'PO1']
for league in leagues:
    df = pd.read_csv(f'csv/values/{league}_players_values_24-25.csv')
    for i in range(len(df)):
        while df.at[i, 'Player'][-1]==' ':
            df.at[i, 'Player'] = df.at[i, 'Player'][:-1]
    print(df.at[0, 'Player'])
    df.to_csv(f'csv/values/{league}_players_values_24-25.csv', index=False)

Simon Mignolet
Carlos Miguel
Ederson
Thibaut Courtois
Gianluigi Donnarumma
Josep Martínez
Jonas Urbig
Timon Wellenreuther
Rui Silva


In [81]:
stats[stats.duplicated(subset=['Player', 'Nation', 'Age'], keep=False)]

,Player,Nation,Pos,Squad,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,PrgR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,xG.1,xAG.1,xG+xAG,npxG.1,npxG+xAG.1,Value


In [366]:
stats[(stats['Squad'].str.contains('Rio'))&(stats['Player'].str.contains(''))]

,Player,Nation,Pos,Squad,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,PrgR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,xG.1,xAG.1,xG+xAG,npxG.1,npxG+xAG.1,Value
211,Vitor Gomes,POR,MF,Rio Ave,36.0,6.0,1.0,93.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.6,0.0,7.0,1.0,0.00,0.97,0.97,0.00,0.97,0.00,0.62,0.62,0.00,0.62,0.1
526,Valentim Sousa,POR,FW,Rio Ave,19.0,5.0,0.0,85.0,0.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.0,0.1,2.0,1.0,7.0,0.00,0.00,0.00,0.00,0.00,0.12,0.00,0.12,0.12,0.12,<NA>
570,Marious Vrousai,GRE,"DF,MF",Rio Ave,26.0,31.0,28.0,2547.0,28.3,1.0,3.0,4.0,1.0,0.0,0.0,5.0,0.0,2.4,2.4,2.1,4.4,47.0,114.0,96.0,0.04,0.11,0.14,0.04,0.14,0.08,0.07,0.16,0.08,0.16,<NA>


In [369]:
values[(values['Squad'].str.contains('Rio'))&(values['Player'].str.contains('Tim'))]

,Player,Value,Squad
296,Tim,0.15,Rio Ave


In [386]:
stats[stats['Value'].isna()]

,Player,Nation,Pos,Squad,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,PrgR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,xG.1,xAG.1,xG+xAG,npxG.1,npxG+xAG.1,Value
40,Ney Bahia,BRA,MF,Santa Clara,20.0,2.0,0.0,62.0,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.0,0.1,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.18,0.0,0.18,0.18,0.18,<NA>


In [17]:
def unite_gk_duplicates(gk_stats):
    stats = copy(gk_stats)
    pairs_to_unite = []
    for i in stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep='last')].index:
        pairs_to_unite.append((i, i+1))
    duplicated_players = stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep=False)]
    stats = stats.drop_duplicates(subset=['Player', 'Age', 'Nation'], keep='last')
    for pair in pairs_to_unite:
        for col in ['MP', 'Starts', 'Min', '90s', 'GA', 'SoTA', 'Saves', 'W', 'D', 'L', 'CS', 'PKatt', 'PKA', 'PKsv', 'PKm',]:
            stats.at[pair[1], col] = duplicated_players.at[pair[0], col] + duplicated_players.at[pair[1], col]
        stats.at[pair[1],'GA90'] = stats.at[pair[1], 'GA'] / stats.at[pair[1], '90s']
        stats.at[pair[1], 'Save%'] = stats.at[pair[1], 'Saves'] / stats.at[pair[1], 'SoTA'] * 100
        stats.at[pair[1], 'CS%'] = stats.at[pair[1], 'CS'] / stats.at[pair[1], 'Starts']
        if stats.at[pair[1], 'PKatt'] != 0:
            stats.at[pair[1], 'Save%.1'] = stats.at[pair[1], 'PKsv'] / (stats.at[pair[1], 'PKatt'] - stats.at[pair[1], 'PKm']) * 100
        for col in stats.columns[5:]:
            stats[col] = stats[col].round(2)
    return stats

In [31]:
for champ_id in champ_ids:
    # stats = pd.read_csv(f'csv/stats/{champ_id}_players_24-25.csv')
    gks = pd.read_csv(f'csv/stats/{champ_id}_gk_24-25.csv')
    # gks = unite_gk_duplicates(gks)
    # for letter in changes:
    #     gks['Player'] = gks['Player'].str.replace(letter, changes[letter])
    # gks = gks.merge(stats[['Player', 'Nation', 'Pos', 'Age', 'Value']], on=['Player', 'Nation', 'Pos', 'Age'], how='inner')
    gks = gks.rename(columns={'Save%.1':'PKsv%'})
    gks['PKsv%'] = gks['PKsv%'].fillna(0)
    gks.to_csv(f'csv/stats/{champ_id}_gk_24-25.csv', index=False)

In [29]:
def unite_misc_duplicates(misc_stats):
    stats = copy(misc_stats)
    pairs_to_unite = []
    for i in stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep='last')].index:
        pairs_to_unite.append((i, i+1))
    duplicated_players = stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep=False)]
    stats = stats.drop_duplicates(subset=['Player', 'Age', 'Nation'], keep='last')
    for pair in pairs_to_unite:
        for col in ['90s', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG']:
            stats.at[pair[1], col] = duplicated_players.at[pair[0], col] + duplicated_players.at[pair[1], col]
    return stats

In [56]:
def unite_shooting_duplicates(shooting_stats):
    stats = copy(shooting_stats)
    pairs_to_unite = []
    for i in stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep='last')].index:
        pairs_to_unite.append((i, i+1))
    duplicated_players = stats[stats.duplicated(subset=['Player', 'Age', 'Nation'], keep=False)]
    stats = stats.drop_duplicates(subset=['Player', 'Age', 'Nation'], keep='last')
    for pair in pairs_to_unite:
        for col in ['90s', 'Gls', 'Sh', 'SoT', 'PK', 'PKatt']:
            stats.at[pair[1], col] = duplicated_players.at[pair[0], col] + duplicated_players.at[pair[1], col]
        if stats.at[pair[1], 'Sh'] > 0:
            stats.at[pair[1], 'SoT%'] = stats.at[pair[1], 'SoT'] / stats.at[pair[1], 'Sh'] * 100
            stats.at[pair[1], 'G/Sh'] = stats.at[pair[1], 'Gls'] / stats.at[pair[1], 'Sh']
        if stats.at[pair[1], 'SoT'] > 0:
            stats.at[pair[1], 'G/SoT'] = stats.at[pair[1], 'Gls'] / stats.at[pair[1], 'SoT']
        stats.at[pair[1], 'Sh/90'] = stats.at[pair[1], 'Sh'] / stats.at[pair[1], '90s']
        stats.at[pair[1], 'SoT/90'] = stats.at[pair[1], 'SoT'] / stats.at[pair[1], '90s']

    stats = stats.fillna(0)
    for col in stats.columns[9:14]:
        stats[col] = stats[col].round(2)
    return stats

In [30]:
for champ_id in champ_ids:
    if champ_id == 'BRA1':
        continue
    stats = pd.read_csv(f'csv/stats/{champ_id}_players_24-25.csv')
    misc = pd.read_csv(f'csv/stats/{champ_id}_misc_24-25.csv')
    misc = unite_misc_duplicates(misc)
    #misc.to_csv(f'csv/stats/{champ_id}_misc_24-25.csv', index=False)
    res = stats.merge(misc[['2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG']], right_index=True, left_index=True)
    res.to_csv(f'csv/stats/{champ_id}_players_24-25.csv', index=False)

In [59]:
for champ_id in champ_ids:
    if champ_id == 'BRA1':
        continue
    stats = pd.read_csv(f'csv/stats/{champ_id}_players_24-25.csv')
    shooting = pd.read_csv(f'csv/stats/{champ_id}_shooting_24-25.csv')
    shooting = unite_shooting_duplicates(shooting)
    shooting.to_csv(f'csv/stats/{champ_id}_shooting_24-25.csv', index=False)
    shooting = pd.read_csv(f'csv/stats/{champ_id}_shooting_24-25.csv')
    res = stats.merge(shooting[['Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT']], how='left', left_index=True, right_index=True)
    res.to_csv(f'csv/stats/{champ_id}_players_24-25.csv', index=False)